# G10: Evaluate a trained model

### How to use (3 steps)
1. In the **Settings** cell below, set `MODEL_NAME`.
2. **Kernel → Restart & Run All.**
3. Scroll to the **bottom: "RESULTS"**. That is the only part you need to read.

Everything in between runs by itself. Charts and detailed tables are **saved to `Models/eval_<MODEL_NAME>/`** instead of filling the screen
(set `SHOW_DETAILS = True` if you want to see them here).

The first run of a model takes a few minutes (it predicts every val/test tile on the GPU). After that the predictions are saved,
and re-running the same model takes seconds.

## Settings (the only cell you edit)

In [ ]:
MODEL_NAME   = "cnn_adamw_v3_long" # "cnn_adamw_v3_long" (final CNN), "mnv2_adamw_v2" (final MobileNetV2), "cnn_adamw_v3"  -> Models/<MODEL_NAME>_best.keras
GPU          = "1"                # which GPU to use (pick the one that is not training)
SHOW_DETAILS = False              # True = also show every per-class table and chart in the notebook

## 1. Setup  *(nothing to change)*

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ["KERAS_BACKEND"]        = "torch"

RUNS     = r"C:\Users\Raphaelo\Documents\Marine Debris Classification\Models"                  # the trained models
EVALS    = os.path.join(RUNS, "Evaluations")   # one eval_<model> folder per model
PREPARED = r"C:\Users\Raphaelo\Documents\Marine Debris Classification\Datasets\prepared"
INPUT, BATCH, WORKERS = 320, 64, 4                      # small batch + few threads = safe next to a training run
OUT   = os.path.join(EVALS, f"eval_{MODEL_NAME}"); os.makedirs(OUT, exist_ok=True)
CACHE = os.path.join(OUT, "predictions.npz")

import csv, json, math, time, collections
import numpy as np, pandas as pd, cv2, torch, keras
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score, roc_curve,
                             multilabel_confusion_matrix, confusion_matrix, accuracy_score)
from IPython.display import display
cv2.setNumThreads(1); torch.set_num_threads(4)
pd.set_option("display.width", 200)

SOURCES = ["seaclear", "bepli", "ucwd", "mados"]
HEAD_OF = {"seaclear": "underwater", "bepli": "shore", "ucwd": "drone", "mados": "satellite"}
HEADS   = ["underwater", "shore", "drone", "satellite"]

def show(x):                       # only displays when SHOW_DETAILS is on
    if SHOW_DETAILS: display(x)
def finish_plot(name):             # always save, only show when SHOW_DETAILS is on
    plt.tight_layout(); plt.savefig(os.path.join(OUT, name), dpi=150)
    plt.show() if SHOW_DETAILS else plt.close()
print("Model:", MODEL_NAME, "| output folder:", OUT)

## 2. Load the test and validation tiles

In [ ]:
classes = json.load(open(os.path.join(PREPARED, "classes.json"), encoding="utf-8"))

def load_manifest(split):
    with open(os.path.join(PREPARED, "manifests", f"{split}.csv"), encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        r["labels"] = [c for c in r["labels"].split(";") if c]
    return rows

val_rows, test_rows = load_manifest("val"), load_manifest("test")
print(f"val {len(val_rows):,} tiles | test {len(test_rows):,} tiles")

## 3. Predict (GPU, slow the first time)
If this model was already evaluated, the saved predictions are reused and the GPU is not touched.
Before using the GPU it checks the card is free, so it stops with a message instead of freezing the PC.

In [ ]:
def read_resized(path):
    img = cv2.cvtColor(cv2.imread(path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    interp = cv2.INTER_AREA if img.shape[0] > INPUT else cv2.INTER_LINEAR
    return cv2.resize(img, (INPUT, INPUT), interpolation=interp).astype(np.float32)

class EvalSet(keras.utils.PyDataset):
    """Every tile once, in order, no augmentation."""
    def __init__(self, rows):
        super().__init__(workers=WORKERS, use_multiprocessing=False, max_queue_size=4)
        self.rows = rows
    def __len__(self):
        return math.ceil(len(self.rows) / BATCH)
    def __getitem__(self, i):
        chunk = self.rows[i * BATCH:(i + 1) * BATCH]
        return np.stack([read_resized(os.path.join(PREPARED, "images", r["path"])) for r in chunk])

paths = lambda rows: np.array([r["path"] for r in rows])
cache_ok = False
if os.path.exists(CACHE):
    z = np.load(CACHE)
    # the cache is only valid for the exact same tiles in the same order (labels files can change, e.g. the v3 relabel)
    cache_ok = ("paths_val" in z.files and np.array_equal(z["paths_val"], paths(val_rows))
                and np.array_equal(z["paths_test"], paths(test_rows)))
    if cache_ok:
        P_val  = {k[4:]: z[k] for k in z.files if k.startswith("val_")}
        P_test = {k[5:]: z[k] for k in z.files if k.startswith("test_")}
        print("Loaded saved predictions (delete predictions.npz to predict again).")
    else:
        print("Saved predictions are for a different set of tiles (manifests changed): predicting again.")
if not cache_ok:
    free, total = (x / 1024**3 for x in torch.cuda.mem_get_info(0))
    print(f"GPU: {torch.cuda.get_device_name(0)} | free {free:.1f} / {total:.1f} GB")
    if free < 6:
        raise RuntimeError(f"Only {free:.1f} GB free on this card: something else is using it. Stopping instead of freezing the PC.")
    torch.cuda.set_per_process_memory_fraction(0.75, 0)   # too big -> clean out-of-memory error, never a freeze
    model = keras.models.load_model(os.path.join(RUNS, f"{MODEL_NAME}_best.keras"), compile=False)
    def predict(rows, name):
        t0 = time.time(); p = model.predict(EvalSet(rows), verbose=1)
        print(f"{name}: {len(rows):,} tiles in {time.time() - t0:.0f} s")
        return {k: np.asarray(v, np.float32) for k, v in p.items()}
    P_val, P_test = predict(val_rows, "val"), predict(test_rows, "test")
    np.savez_compressed(CACHE, **{f"val_{k}": v for k, v in P_val.items()}, **{f"test_{k}": v for k, v in P_test.items()},
                        paths_val=paths(val_rows), paths_test=paths(test_rows))

## 4. Fine-grained results (which kind of debris?)
- Each head is scored **only on its own dataset's tiles** (same masking as training).
- One threshold per class is tuned on **validation** (best F1), then applied once to **test**.
- **Precision**: of the tiles flagged, how many were right. **Recall**: of the real ones, how many were found.
  **F1**: balance of both. **ROC-AUC**: how well the scores rank, independent of any threshold.

In [ ]:
GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)

def head_data(rows, P, head):
    idx = [i for i, r in enumerate(rows) if HEAD_OF[r["source"]] == head]
    ci = {c: j for j, c in enumerate(classes[head])}
    Y = np.zeros((len(idx), len(ci)), np.int32)
    for k, i in enumerate(idx):
        for c in rows[i]["labels"]: Y[k, ci[c]] = 1
    return Y, P[head][idx], idx

def best_threshold(y, p, grid=GRID):
    if y.sum() == 0: return 0.5
    f1s = [precision_recall_fscore_support(y, p >= t, average="binary", zero_division=0)[2] for t in grid]
    return float(grid[int(np.argmax(f1s))])

def evaluate(Y, Pr, t, cls):
    Yh = (Pr >= t).astype(int)
    p, r, f, sup = precision_recall_fscore_support(Y, Yh, average=None, zero_division=0)
    aucs = [roc_auc_score(Y[:, j], Pr[:, j]) if 0 < Y[:, j].sum() < len(Y) else np.nan for j in range(len(cls))]
    per = pd.DataFrame({"class": cls, "support": sup, "precision": p, "recall": r, "f1": f, "roc_auc": aucs,
                        "accuracy": (Yh == Y).mean(0), "threshold": t})
    mp, mr, mf, _ = precision_recall_fscore_support(Y, Yh, average="macro", zero_division=0)
    up, ur, uf, _ = precision_recall_fscore_support(Y, Yh, average="micro", zero_division=0)
    summ = {"macro_precision": mp, "macro_recall": mr, "macro_f1": mf, "macro_roc_auc": np.nanmean(aucs),
            "micro_precision": up, "micro_recall": ur, "micro_f1": uf,
            "label_accuracy": (Yh == Y).mean(), "exact_match_accuracy": (Yh == Y).all(1).mean()}
    return per, summ

data = {s: {h: head_data(rows, P, h) for h in HEADS}
        for s, rows, P in [("val", val_rows, P_val), ("test", test_rows, P_test)]}
thr = {h: np.array([best_threshold(data["val"][h][0][:, j], data["val"][h][1][:, j]) for j in range(len(classes[h]))])
       for h in HEADS}
json.dump({h: dict(zip(classes[h], thr[h].tolist())) for h in HEADS}, open(os.path.join(OUT, "thresholds.json"), "w"), indent=2)

per_class, summaries = {}, {}
for h in HEADS:
    Y, Pr, _ = data["test"][h]
    per_class[h], summaries[h] = evaluate(Y, Pr, thr[h], classes[h])
    _, summaries[h + " (threshold 0.5)"] = evaluate(Y, Pr, np.full(len(classes[h]), 0.5), classes[h])
    per_class[h].to_csv(os.path.join(OUT, f"test_{h}_per_class.csv"), index=False)
    if SHOW_DETAILS:
        print(f"\n=== {h} ({len(Y):,} test tiles) ==="); display(per_class[h].round(3))
summary_df = pd.DataFrame(summaries).T
summary_df.to_csv(os.path.join(OUT, "test_summary.csv"))
print("Saved per-class tables and test_summary.csv")

## 5. Viewpoint head (which camera type?)

In [ ]:
VIEWS = ["underwater", "shore", "drone", "satellite"]; VIEW_IDX = {s: i for i, s in enumerate(SOURCES)}
yv = np.array([VIEW_IDX[r["source"]] for r in test_rows]); pv = P_test["viewpoint"].argmax(1)
viewpoint_acc = accuracy_score(yv, pv)
cmv = confusion_matrix(yv, pv)
fig, ax = plt.subplots(figsize=(5, 4.3))
im = ax.imshow(cmv / cmv.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cmv[i, j]:,}", ha="center", va="center", color="white" if cmv[i, j] > cmv[i].sum() / 2 else "black")
ax.set_xticks(range(4), VIEWS, rotation=30); ax.set_yticks(range(4), VIEWS)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Viewpoint: {MODEL_NAME}"); plt.colorbar(im, fraction=.046)
finish_plot("viewpoint_confusion.png")
print(f"Viewpoint accuracy: {viewpoint_acc:.4f}")

## 6. Charts: confusion matrices and ROC curves *(saved to the folder)*

In [ ]:
for h in HEADS:
    Y, Pr, _ = data["test"][h]; Yh = (Pr >= thr[h]).astype(int)
    mcm = multilabel_confusion_matrix(Y, Yh); n = len(classes[h]); cols = 4; rws = math.ceil(n / cols)
    fig, axs = plt.subplots(rws, cols, figsize=(3.3 * cols, 3.0 * rws))
    for k, ax in enumerate(np.atleast_1d(axs).ravel()):
        if k >= n: ax.axis("off"); continue
        m = mcm[k]; norm = m / m.sum(1, keepdims=True).clip(1)
        ax.imshow(norm, cmap="Greens", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{m[i, j]:,}", ha="center", va="center", fontsize=10, color="white" if norm[i, j] > 0.5 else "black")
        ax.set_xticks([0, 1], ["no", "yes"]); ax.set_yticks([0, 1], ["no", "yes"])
        ax.set_title(classes[h][k][:24], fontsize=9); ax.set_xlabel("predicted", fontsize=8); ax.set_ylabel("true", fontsize=8)
    fig.suptitle(f"{h}: test confusion matrices ({MODEL_NAME})"); finish_plot(f"confusion_{h}.png")

fig, axs = plt.subplots(1, 4, figsize=(22, 5.2))
for ax, h in zip(axs, HEADS):
    Y, Pr, _ = data["test"][h]
    for j, c in enumerate(classes[h]):
        if 0 < Y[:, j].sum() < len(Y):
            fpr, tpr, _ = roc_curve(Y[:, j], Pr[:, j])
            ax.plot(fpr, tpr, lw=1.2, label=f"{c[:18]} ({roc_auc_score(Y[:, j], Pr[:, j]):.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_title(f"{h}: macro AUC {summaries[h]['macro_roc_auc']:.3f}")
    ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.legend(fontsize=6.5, loc="lower right")
fig.suptitle(f"Test ROC curves: {MODEL_NAME}"); finish_plot("roc_curves.png")
print("Saved confusion_*.png and roc_curves.png")

## 7. Photo level
Tiles are put back into their original photo: a photo has a class if **any** of its tiles does (highest score across its tiles).
Thresholds are tuned again at photo level on validation. MADOS patches are already whole images, so satellite is unchanged.

In [ ]:
def photo_level(rows, P, head):
    groups = collections.defaultdict(list)
    for i, r in enumerate(rows):
        if HEAD_OF[r["source"]] == head: groups[r["orig"]].append(i)
    ci = {c: j for j, c in enumerate(classes[head])}
    Y = np.zeros((len(groups), len(ci)), int); Pr = np.zeros((len(groups), len(ci)), np.float32)
    for k, ids in enumerate(groups.values()):
        Pr[k] = P[head][ids].max(0)
        for i in ids:
            for c in rows[i]["labels"]: Y[k, ci[c]] = 1
    return Y, Pr

photo = {}
for h in HEADS:
    Yv, Pv = photo_level(val_rows, P_val, h); Yt, Pt = photo_level(test_rows, P_test, h)
    tp = np.array([best_threshold(Yv[:, j], Pv[:, j]) for j in range(len(classes[h]))])
    _, s = evaluate(Yt, Pt, tp, classes[h]); s["photos"] = len(Yt); photo[h] = s
photo_df = pd.DataFrame(photo).T
photo_df.to_csv(os.path.join(OUT, "test_photo_level.csv"))
show(photo_df.round(3)); print("Saved test_photo_level.csv")

## 8. Trash vs clean (is there **any** debris in the tile?)
- **Truth**: the tile has at least one trash label.
- **Score**: combines all of the head's class scores with *noisy-OR*: `1 - (1-p1)(1-p2)...` = the chance that at least one class is present.
  (Tested on saved predictions: it ranks better than taking the single highest score, especially underwater.)
- Satellite: most MADOS classes are not trash (ships, platforms, algae, waves), so trash = **Marine Debris** only;
  a second line also counts **Oil Spill**.
- One threshold per head, tuned on validation, applied once to test.
- **"F1 if always yes"** is what a useless model that always answers "trash" would score. Compare against it:
  when most tiles have trash (shore), a high F1 can be mostly free.

In [ ]:
TRASH = {h: classes[h] for h in ["underwater", "shore", "drone"]}
TRASH["satellite"] = ["Marine Debris"]
TRASH["satellite + oil spill"] = ["Marine Debris", "Oil Spill"]
BIN_GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)

def binary(rows, P, name):
    head = name.split(" ")[0]; Y, Pr, _ = head_data(rows, P, head)
    cols = [classes[head].index(c) for c in TRASH[name]]
    return Y[:, cols].max(1), 1 - np.prod(1 - Pr[:, cols], axis=1)

binary_res = {}
for name in TRASH:
    yv, sv = binary(val_rows, P_val, name); yt, st = binary(test_rows, P_test, name)
    t = best_threshold(yv, sv, BIN_GRID); yh = (st >= t).astype(int)
    p, r, f, _ = precision_recall_fscore_support(yt, yh, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    share = yt.mean()
    binary_res[name] = {"tiles": len(yt), "trash_share": share, "roc_auc": roc_auc_score(yt, st),
                        "precision": p, "recall": r, "f1": f, "f1_if_always_yes": 2 * share / (1 + share),
                        "accuracy": (yh == yt).mean(), "threshold": t, "TP": tp, "FP": fp, "FN": fn, "TN": tn}
binary_df = pd.DataFrame(binary_res).T
binary_df.to_csv(os.path.join(OUT, "test_binary_tile.csv"))
show(binary_df.round(3)); print("Saved test_binary_tile.csv")

# v3 models also have a dedicated "trash present?" neuron per head (<head>_trash), trained directly on this question.
# Satellite's neuron was trained on Marine Debris OR Oil Spill, so it is scored against that truth.
neuron_res = {}
for h in HEADS:
    if f"{h}_trash" not in P_test: continue
    name = "satellite + oil spill" if h == "satellite" else h
    yv, _ = binary(val_rows, P_val, name); yt, _ = binary(test_rows, P_test, name)
    _, _, iv = head_data(val_rows, P_val, h); _, _, it = head_data(test_rows, P_test, h)
    sv, st = P_val[f"{h}_trash"][iv, 0], P_test[f"{h}_trash"][it, 0]
    t = best_threshold(yv, sv, BIN_GRID); yh = (st >= t).astype(int)
    p, r, f, _ = precision_recall_fscore_support(yt, yh, average="binary", zero_division=0)
    neuron_res[h] = {"truth": name, "roc_auc": roc_auc_score(yt, st), "precision": p, "recall": r, "f1": f,
                     "f1_if_always_yes": 2 * yt.mean() / (1 + yt.mean()), "accuracy": (yh == yt).mean(), "threshold": t}
if neuron_res:
    neuron_df = pd.DataFrame(neuron_res).T
    neuron_df.to_csv(os.path.join(OUT, "test_binary_neuron.csv"))
    show(neuron_df); print("Saved test_binary_neuron.csv (dedicated trash neuron)")

---
# RESULTS
Test set only. **AUC** = ranking quality (0.5 = guessing, 1 = perfect). **F1** = quality of the actual yes/no decisions.

In [ ]:
print(f"MODEL: {MODEL_NAME}        Viewpoint accuracy: {viewpoint_acc:.1%}\n")
res = pd.DataFrame({
    "AUC (kind of debris)":   {h: summaries[h]["macro_roc_auc"] for h in HEADS},
    "F1 per tile":            {h: summaries[h]["macro_f1"] for h in HEADS},
    "F1 per photo":           {h: photo[h]["macro_f1"] for h in HEADS},
    "Trash/clean AUC":        {h: binary_res[h]["roc_auc"] for h in HEADS},
    "Trash/clean F1":         {h: binary_res[h]["f1"] for h in HEADS},
    "F1 if always 'trash'":   {h: binary_res[h]["f1_if_always_yes"] for h in HEADS},
})
if neuron_res:   # v3 models: the dedicated trash neuron, next to the trash score combined from the class outputs
    res["Trash neuron AUC"] = {h: neuron_res[h]["roc_auc"] for h in HEADS}
    res["Trash neuron F1"]  = {h: neuron_res[h]["f1"] for h in HEADS}
display(res.round(3))
if neuron_res:
    print("Trash neuron: satellite is scored as Marine Debris OR Oil Spill (what it was trained on).")
print("Satellite trash = Marine Debris only. With Oil Spill counted too: "
      f"AUC {binary_res['satellite + oil spill']['roc_auc']:.3f}, F1 {binary_res['satellite + oil spill']['f1']:.3f}")

### All evaluated models side by side

In [ ]:
rows = []
for d in sorted(os.listdir(EVALS)):
    f = os.path.join(EVALS, d, "test_summary.csv")
    if not (d.startswith("eval_") and os.path.exists(f)): continue
    s = pd.read_csv(f, index_col=0); b = os.path.join(EVALS, d, "test_binary_tile.csv")
    b = pd.read_csv(b, index_col=0) if os.path.exists(b) else None
    nf = os.path.join(EVALS, d, "test_binary_neuron.csv"); nn = pd.read_csv(nf, index_col=0) if os.path.exists(nf) else None
    for h in HEADS:
        rows.append({"model": d[5:], "head": h, "AUC": s.loc[h, "macro_roc_auc"], "F1 per tile": s.loc[h, "macro_f1"],
                     "Trash/clean AUC": b.loc[h, "roc_auc"] if b is not None else np.nan,
                     "Trash neuron AUC": nn.loc[h, "roc_auc"] if nn is not None else np.nan})
cmp = pd.DataFrame(rows)
for metric in ["AUC", "F1 per tile", "Trash/clean AUC", "Trash neuron AUC"]:
    print(metric); display(cmp.pivot(index="head", columns="model", values=metric).round(3))
print("Empty cells: older evaluations (re-run them with this notebook) / models without a trash neuron (before v3).")